## RL Methods: Monte Carlo

- We've covered the DP approach, which looks at the fairly restrictive case of solving the Bellman Equations when the full MDP is known; i.e. you know both the transition probability, and the reward. This is typically unrealistic. Very few process in reality will be kind enough to provide you both pieces of information perfectly. And even if you estimate both using data, it will be estimated under some uncertainty

- Therefore, we need more general methods to solve for the optimal policy when the transition probabilities and rewards are not known

- Here, we consider the Monte-Carlo approach. 

- Note that the traditional Monte-Carlo approach assumes an **episodic task**. That is, the iteration will run for some time, and reach a terminal point.
    - You can manually impose a terminal point for Monte-Carlo, but note that this isn't a true Monte Carlo approach, but a **truncated** Monte Ccarlo

- Idea:
    - The idea of the monte carlo is to simply let entire episodes play out multiple times, to find the long term average state-value function and hence the optimal policy
    - So you have a `step()` method that determines 2 things; (i) what is the next state you are in, and (ii) is this next state a terminal one?
    - Having done this, you now have 2 series recorded (i) a series of states, ending with a terminal state, and (ii) a series of rewards, indicating the reward you get for landing on that state
        - Note that the series of rewards is NOT the sum of your future rewards (i.e. not the true Q-value). It is the specific reward of reaching some state $S$
    - Now, for every point we land on in our array, we want to compute the discounted future rewards observed from that point, so we can derive a proper state value
    - Init an array to hold the state-value function $V$
    - We'll compute this in reverse order of the rewards array;
        - Init $G$ as the running total of the discounted rewards. This starts at 0
        - We know that the last state is terminal. Suppose this is the rightmost terminal state. 
            - We know that reaching this state gives a reward of 1. 
            - We also know that, this being the terminal state, there is no future reward. 
            - So the cumulative future reward $G = 1$
            - Since we know the rightmost state $N$ has value $G=1$, update its state-value function $V[N]$ by taking $V[N] + \alpha[G - V[N]]$
        - Moving to the second last state. 
            - We know that this state has reward 0. Therefore, the value of this state comes only from the discounted value of transitioning to the last state and receiving the terminal reward 1. 
            - Thus, update $G = reward[N-1] + \gamma \cdot G$
            - Then, update the state value function $V[N-1] + \alpha[G - V[N-1]]$
        - Iterate until you reach the end of the episode 
    - Continue to the next episode for a fixed number of episodes

In [ ]:
import numpy as np
np.random.seed(0)

# Environment setup; we create a number line with 10 states. The right-most state has a reward of 1
# Therefore, train the RL loop to take rightward actions [+1] more than leftward actions [-1]
# Terminate if you reach the left or rightmost states
N_STATES = 5
N_ACTIONS = 5
GAMMA = 0.9
ALPHA = 0.1
EPSILON = 0.5
EPISODES = 1000
TD_N = 3

TRANSITION_PROBABILITIES = np.random.rand(N_STATES, N_ACTIONS, N_STATES)
TRANSITION_PROBABILITIES /= np.sum(TRANSITION_PROBABILITIES, axis=2, keepdims=True)
REWARDS = np.random.rand(N_STATES, N_ACTIONS, N_STATES) * 5

In [ ]:
from collections import namedtuple

episode_step = namedtuple('episode_step', ['curr_state', 'action', 'new_state', 'reward'])

def take_action(curr_state: int, action_values: np.ndarray) -> tuple[int]:
    action_values_for_curr_state = action_values[curr_state, :]
    
    if np.random.rand() >= EPSILON:
        highest_action_value_action = np.argmax(action_values_for_curr_state)
        return highest_action_value_action
    else:
        random_action = np.random.choice(N_ACTIONS)
        return random_action

def monte_carlo_epsilon_greedy():
    action_values = np.zeros((N_STATES, N_ACTIONS))
    state_values = np.zeros(N_STATES)

    rewards_intermediate_store = {
        (s,a): [] for s in range(N_STATES) for a in range(N_ACTIONS)
    }
    for _ in range(EPISODES):
        curr_state = np.random.choice(list(range(1,N_STATES-1)))
        episode_history = []

        terminal_state_reached = curr_state in [0, N_STATES-1]
        while not terminal_state_reached:
            action = take_action(curr_state, action_values)
            new_state = np.random.choice(
                N_STATES, p=TRANSITION_PROBABILITIES[curr_state, action]
            )
            terminal_state_reached = new_state in [0, N_STATES-1]
            reward = REWARDS[curr_state, action, new_state]
            episode_history.append(
                episode_step(curr_state=curr_state, action=action, new_state=new_state, reward=reward)
            )
            curr_state = new_state
        
        return_val = 0
        for step in reversed(episode_history):
            return_val = step.reward + GAMMA * return_val
            rewards_intermediate_store[(step.curr_state, step.action)].append(return_val)
        
        for curr_state, action in rewards_intermediate_store.keys():
            action_values[curr_state, action] = np.mean(rewards_intermediate_store.get((curr_state, action))) if rewards_intermediate_store.get((curr_state, action)) != [] else 0
        
        state_values = np.max(action_values, axis=1)
    
    return state_values, action_values


In [ ]:
monte_carlo_epsilon_greedy()